In [ ]:
import json
import os
from pathlib import Path
import time

from dotenv import load_dotenv
import geopandas as gpd
import pandas as pd
import plotly.express as px
import plotly.io as pio
import pycountry
import requests
from shapely.geometry import Point

In [ ]:
# Folder with WIA in Sharepoint
base_dir = "./"
data_out_folder = base_dir + "data_out/"
data_in_folder = base_dir + "data_in/"

# out folder for maps (hazards)
maps_hazards_folder = base_dir + "data_out/maps/hazards/"

# shapes subfolder
data_in_shapes = base_dir + "data_in/shapes/"

# out folder for maps (population)
maps_pop_folder = base_dir + "data_out/maps/PIN_population/"

# out folder for templates (pcodes standardized)
templates_out_folder = base_dir + "data_out/templates/"


In [ ]:
# NOTE: selected countries for WIA execution
wia_countries_exercise = [
    # "Kenya",
    # "Mali",
    # "Benin",
    # "Lebanon",
    # "Togo",
    # "Afghanistan",
    # "Ukraine",
    # "Burkina Faso",
    # "Niger",
    # "Honduras",
    # "El Salvador",
    # "Cameroon",
    "Central African Republic",
    # "Myanmar",
    # "South Sudan",
    # "Syria",
    # "Ethiopia",
    # "Congo, The Democratic Republic of the",
    # "Haiti",
    # "Somalia",
    # "Sudan",
    # "Yemen",
    # "Saint Vincent and the Grenadines",
    # "Grenada",
]
country = wia_countries_exercise[0]
c_iso3 = (
    pycountry.countries.search_fuzzy(country)[0].alpha_3 if country != "Niger"
    # introduced Niger exception (known case to avoid Nigeria)
    else pycountry.countries.search_fuzzy(country)[1].alpha_3
)

# NOTE: select admin level
admin_level = 2

# Parent P-Codes column/values
# NOTE: admin0 (national) not in parent_admins (treated separately, if exists)
parent_admins = list(range(1, admin_level))


In [ ]:
# column pcode and name (assume harmonized across CODs)
pcode_col = f"adm{admin_level}_pcode"
name_col = f"adm{admin_level}_name"
parent_pcode_cols = [f"adm{p_a}_pcode" for p_a in parent_admins]

# read templates file (or files)
template_dfs = [
    pd.read_excel(
        templates_out_folder + f"{c_iso3}_adm{a_l + 1}_Pcodes_Template.xlsx"
    ).astype(str)
    for a_l in parent_admins
] if parent_admins else [
    pd.read_excel(
        templates_out_folder + f"{c_iso3}_adm{admin_level}_Pcodes_Template.xlsx"
    ).astype(str)
]

# geojson output file name (or names)
geo_filenames = [
    f"geojson_{c_iso3}_adm{a_l + 1}"
    for a_l in parent_admins
] if parent_admins else [
    f"geojson_{c_iso3}_adm{admin_level}"
]
# Get geojson/s in shapes
geojson_files = [
    f.name
    for f in Path(data_in_shapes).iterdir()
    if f.name.endswith(".zip") and any(g_f in f.name for g_f in geo_filenames)
]

# read country shape (or shapes) geojson/s into gdf
gdfs = [
    gpd.read_file(
    data_in_shapes + f"{g_f}"
    ).to_crs('EPSG:4326') for g_f in geojson_files
    for g_f in geojson_files
]


In [ ]:
# Use of IDMC API request
# GRID Report spatially disaggregated dataset with geolocations
def make_api_call(url, max_attempts=3, sleep_duration=1, **kwargs):
    """
    Makes an API call with retry logic.

    :param url: URL for the API call.
    :param max_attempts: Maximum number of attempts for the API call.
    :param sleep_duration: Duration (in seconds) to sleep between attempts.
    :param kwargs: Additional arguments to pass to the requests.get() method.
    :return: Response object if successful, None otherwise.
    """
    for attempt in range(max_attempts):
        try:
            response = requests.get(url, **kwargs)
            response.raise_for_status()
            return response  # Return the successful response

        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1} of {max_attempts} failed: {e} in url: {url}")
            if attempt < max_attempts - 1:
                time.sleep(sleep_duration)  # Wait before retrying

    print(f"Failed to retrieve data from {url} after {max_attempts} attempts.")
    return None


In [ ]:
# read .env
load_dotenv()  

# access idmc token from .env file
idmc_client_id = os.environ["aps_idmc_client_id"]

# GIDD disaggregated data endpoint (latest published GRID report)
# quality-controlled annually validated data on internal displacement
# By conflicts and disasters, disaggregated by caseload, location and event.
url_end = "https://helix-tools-api.idmcdb.org/external-api/gidd/disaggregations/disaggregation-geojson/"

# there's no filter by country, all geojson download
idmc_response = make_api_call(url_end, params={"client_id": idmc_client_id})
json_data = idmc_response.json()


In [ ]:
# make a dataframe and explode with geojson features property and geometry
# (pandas handle lists as inputs, geopandas read_file geojson no)
idmc_df = pd.DataFrame(
  [
    {**f["properties"], **f["geometry"]}
    for f in json_data["features"]
  ]
).explode(
    column=[
        "Locations name",
        "Locations accuracy",
        "Locations type",
        "coordinates",
    ]
)


##### Exploratory data analysis for the specific country
- Filter IDPs by selected Country (year should be the latest GRID report)
- Verify if figures match GRID report
    - If not, there should be many-to-one relationships or multiple location types (duplicated IDs)
- Check which location accuracy is the most representative at subnational level (e.g. ADM1 or ADM2)
    - Also check if the selected `admin_level` is align with the above analysis
- Check the available location types
    - Location type is disregarded if unique (assumes Origin and Destination have the same impact or are the same)
    - If not unique, Destination is used for analysis under the assumption of pressure to WASH systems

In [ ]:
# build query for the specific selected country and year
fig_cat = "Internal Displacements"
# Use last reporting year
rep_year = idmc_df.Year.max()

# filter IDPs at subnational level for selected country
country_idp_df = idmc_df.query(
    # NOTE: some countries report admin0 figures (no spatial granularity)
    # E.g. El Salvador (estimated by University) and Honduras conflict IDPs
    # Treat those "`Locations accuracy`.str.contains('AM0')" particularly
    "ISO3 == @c_iso3 & `Figure category` == @fig_cat & Year == @rep_year"
).reset_index(drop=True)

# join values in list values of sources column
country_idp_df["expand_source"] = country_idp_df.Sources.apply(lambda x: ", ".join(x))

# figure cause/s
fig_cause = country_idp_df["Figure cause"].unique()
# count subnational representation
sub_nat_rep = country_idp_df["Locations accuracy"].value_counts()
# count of locations type representation
loc_type_rep = country_idp_df["Locations type"].value_counts()

# print simple EDA stats (figures rounded should match GRID)
print(
    f"Total figures {c_iso3} {rep_year} by {list(fig_cause)}: {country_idp_df['Total figures'].sum()}"
)

# if figures don't verify with GRID report there should be many-to-one relationships or multiple location types
print(
    f"\nNumber of many-to-one IDs: {country_idp_df.duplicated('ID').sum()}"
)

# subnational representation counts
print(f"\nValue counts at subnat level to verify consistency with selected 'admin{admin_level}' analysis:\n{sub_nat_rep}")

# value counts of the location types to analyze
print(f"\nValue counts of location types:\n{loc_type_rep}")


##### Geocoding of location points
- Geocode against OCHA-COD shapes for the selected country
<!--
- Merge geocoding with OCHA-COD shapes using names
    - Very good agreement so far found between WFP shapes in GeoRepo with COD names
- Drop all IDs from geocoded Destinations and proceed with other Locations type
    - These includes e.g. those without proper Origin/Destination distinction
    - To be used only if those locations type are present (TODO: error handling)
-->

<!--
- Starts with Unique Destination
    - To be used only if Destination points present (TODO: error handling)
- Check that location accuracy is above the selected subnational level of analisis `admin_level`
- Drop all IDs from geocoded Destinations and proceed with other Locations type
    - These includes e.g. those without proper Origin/Destination distinction
    - To be used only if those locations type are present (TODO: error handling)
-->

In [ ]:
# create dictionary/ies of polygons from country admin_level (or levels) geodataframe
polygons = [
    dict(zip(gdf[p_c], gdf['geometry']))
    for p_c, gdf in zip(
        [
            f"adm{p_a + 1}_pcode" for p_a in parent_admins
        ] if parent_admins else [pcode_col],
        gdfs,
    )
]

# get_point_in_polygon function: lat/lon coordinates --> pcode (admin area containing point)
# Function with type hints: expects polygons dictionary with id key and geometry value
def get_point_in_polygon(lat: float, lon: float, polygons: dict) -> str:
    """ 
        @param lat: double
        @param lon: double
        @param polygons: dict
        @return pcode: str
    """
    point = Point(lon, lat)
    for pcode in polygons:
        polygon = polygons[pcode]
        if polygon.contains(point):
            return pcode
    return 'null'


In [ ]:
# get_closest_polygon function: lat/lon coordinates --> pcode (admin area with the closest boundary)
# Function with type hints: expects polygons dictionary with id key and geometry value
def get_closest_polygon(lat: float, lon: float, polygons: dict) -> str:
    """ 
    Returns the pcode of the polygon closest to the given point
        @param lat: double
        @param lon: double
        @param polygons: dict
        @return pcode: str
    """
    point = Point(lon, lat)
    return min(
        ((polygon.distance(point), pcode) for pcode, polygon in polygons.items()),
        key=lambda x: x[0]
    )[1]


In [ ]:
# use get_point_in_polygon to determine which admin area contains the geolocation
# loop across number of parent admin levels (one or more)
for i, p_c in enumerate(
    [
        f"adm{p_a + 1}_pcode" for p_a in parent_admins
    ] if parent_admins else [pcode_col]
):
    print(f"Determining {p_c} for geo locations")
    country_idp_df[p_c] = country_idp_df.apply(
        lambda x: get_point_in_polygon(
            x['coordinates'][1],
            x['coordinates'][0],
            polygons[i],
        ),
        axis=1,
    )

    # review if null pcode values (gelocation outside admin shapes)
    mask_null_pcodes = country_idp_df[p_c] == 'null'
    n_nulls = mask_null_pcodes.sum()
    if n_nulls > 0:
        print(f"REVISE {n_nulls} entries WITHOUT pcode")
        # apply get_closest_polygon for the null pcodes (determine closest admin area)
        country_idp_df.loc[
            mask_null_pcodes,
            p_c
        ] = country_idp_df.loc[mask_null_pcodes].apply(
            lambda x: get_closest_polygon(
                x['coordinates'][1],
                x['coordinates'][0],
                polygons[i],
            ),
            axis=1,
        )


##### Analysis for Destination points
- Assumes location accuracy `Point`, if available, is at the level of analysis
- Check that location accuracy is above or equal the selected subnational level of analisis `admin_level`
  - If not, those values belong to `parent_admins` and must be treated separately
  - If `admin_level` higher than 2, `len(parent_admins) > 1`: a corresponding loop takes care accordingly
  - Nationally reported figures (`admin0`) if any are equally distributed by pop at workflow end
<!--
- To be used only if Destination points present
  - TODO: error handling (actually works for missing Destination points)
- Differentiate many-to-one relationships
- Merge geocoding with OCHA-COD shapes using names
  - Very good agreement so far found between WFP shapes in GeoRepo with COD names
- Drop all IDs from geocoded Destinations and proceed with other Locations type
  - These includes e.g. those without proper Origin/Destination distinction
  - To be used only if those locations type are present (TODO: error handling)
-->

<!--
- Starts with Unique Destination
  - To be used only if Destination points present (TODO: error handling)
- Check that location accuracy is above the selected subnational level of analisis `admin_level`
- Drop all IDs from geocoded Destinations and proceed with other Locations type
  - These includes e.g. those without proper Origin/Destination distinction
  - To be used only if those locations type are present (TODO: error handling)
-->

In [ ]:
# first analysis: identify destination points
country_idp_dest_df = country_idp_df.query(
    "`Locations type` == 'Destination'"
).reset_index(drop=True)

# identify flow IDs with destination
dest_ids = country_idp_dest_df.ID.unique()

# Destination points counts of location accuracy
dest_count_series = country_idp_dest_df["Locations accuracy"].value_counts()

# subnational representation for destination points dataframe
print(
    f"Value counts of subnat levels in Destination points Dataframe:\n{dest_count_series}"
)

# verify there are no points below the selected representation level
dest_loc_above = [
    l_a
    for a_l in ([f"ADM{p_a}" for p_a in parent_admins] + ["AM0"])
    for l_a in dest_count_series.index
    if a_l in l_a
]
# NOTE: "AM0", national figures, treated separately if reported
dest_loc_above_no_nat = [l_a for l_a in dest_loc_above if "AM0" not in l_a]

if dest_loc_above:
    print(f"\nREVISE that {list(dest_count_series.index)} contains level(s) below admin{admin_level} and FILTER accordingly")
    
    # keep lower resolution (LR) flows with destination for later figures distribution
    country_idp_dest_LR_df = country_idp_dest_df.query(
        "`Locations accuracy` in @dest_loc_above_no_nat"
    ).reset_index(drop=True)

    # filter displacement flows below the selected representation level
    country_idp_dest_df = country_idp_dest_df.query(
        "`Locations accuracy` not in @dest_loc_above"
    ).reset_index(drop=True)

    # verification of filter
    dest_filtered_count_series = country_idp_dest_df["Locations accuracy"].value_counts()
    # subnational representation of destination points filtered dataframe
    print(
        f"\nValue counts of subnat levels in Filtered Destination Dataframe:\n{dest_filtered_count_series}"
    )


##### Analysis of other Locations type
- Drop all Destination IDs and proceed with other Locations type
  - Assumes there are e.g. cases without proper Origin/Destination distinction
- Assumes accuracy named `Point`, if available, is at the level of analysis
- Check that location accuracy is above or equal the selected subnational level of analisis `admin_level`
  - If not, those values belong to `parent_admins` and must be treated separately
  - If `admin_level` higher than 2, `len(parent_admins) > 1`: a corresponding loop takes care accordingly
  - Nationally reported figures (`admin0`) if any are equally distributed by pop at workflow end
  <!-- - TODO: error handling (actually works for missing Destination points) -->

In [ ]:
# filter displacement flows not in destination IDs (including those with lower resolution)
country_idp_org_df = country_idp_df.query(
    "~ID.isin(@dest_ids)"
).reset_index(drop=True)

# Origin points counts of location accuracy
org_count_series = country_idp_org_df["Locations accuracy"].value_counts()

# count of locations type representation in origin dataframe
loc_type_org_rep = country_idp_org_df["Locations type"].value_counts()

# subnational representation for origin points dataframe
print(
    f"Value counts of subnat levels in Origin points Dataframe:\n{org_count_series}"
)

# value counts of the location types to analyze in origin dataframe
print(f"\nValue counts of location types in Origin points Dataframe:\n{loc_type_org_rep}")

# verify there are no points below the selected representation level (origin dataframe)
org_loc_above = [
    l_a
    for a_l in ([f"ADM{p_a}" for p_a in parent_admins] + ["AM0"])
    for l_a in org_count_series.index
    if a_l in l_a
]
# NOTE: "AM0", national figures, treated separately if reported
org_loc_above_no_nat = [l_a for l_a in org_loc_above if "AM0" not in l_a]

if org_loc_above:
    print(f"\nREVISE that {list(org_count_series.index)} contains level(s) below admin{admin_level} and FILTER accordingly")

    # keep lower resolution (LR) flows with origins for later figures distribution
    country_idp_org_LR_df = country_idp_org_df.query(
        "`Locations accuracy` in @org_loc_above_no_nat"
    ).reset_index(drop=True)

    # filter displacement flows below the selected representation level
    country_idp_org_df = country_idp_org_df.query(
        "`Locations accuracy` not in @org_loc_above"
    ).reset_index(drop=True)

    # verification of filter
    org_filtered_count_series = country_idp_org_df["Locations accuracy"].value_counts()
    # subnational representation of origin points filtered dataframe
    print(
        f"\nValue counts of subnat levels in Filtered Origin Dataframe:\n{org_filtered_count_series}"
    )


##### Analysis of Figures
- Merge Destination with Origin dataframes
  - Origin is used to account displacements where no Origin/Destination difference is reported
- Duplicated IDs are solved using Population Figures:
  - Brings population of the `rep_year` (WorldPop)
  - Divide total figures of repeated IDs based on population
<!-- - Checkout if subnational figures add up to GRID report
  - Note if levels below `admin_level` were FILTERED before and distribute figures using even-population approach -->

In [ ]:
# read population data using selected country/admin level/year accordingly
pop_df = pd.read_excel(
    maps_pop_folder + f"{c_iso3}_adm{admin_level}_{rep_year}_WorldPop.xlsx"
).astype({pcode_col: str})

# merge population to idmc data at the geocoded admin level
country_idp_pop_df = pd.concat(
    [country_idp_dest_df, country_idp_org_df],
    ignore_index=True,
).merge(
    pop_df,
    how="left",
    on=pcode_col,
    validate="m:1",
)


In [ ]:
# merge country_idp_pop_df with population sums at IDs (for figure distribution at duplicated IDs)
country_idp_pop_df = country_idp_pop_df.merge(
    country_idp_pop_df.groupby("ID", as_index=False).agg(
        pop_sum_at_ID_admin_level=(f"tot_pop_{rep_year}", "sum"),
    ),
    how="inner",
    on="ID",
    validate="many_to_one",
)

# compute pop weight in new column
country_idp_pop_df["pop_w"] = (
    country_idp_pop_df[f"tot_pop_{rep_year}"] / country_idp_pop_df.pop_sum_at_ID_admin_level
)

# use pop weight to distribute reported figures with duplicated IDs
country_idp_pop_df["ID_pop_w"] = (
    country_idp_pop_df.pop_w * country_idp_pop_df["Total figures"]
)


##### Analysis of *Low Resolution* Figures
- Merge Destination with Origin dataframes for low resolution (if both exists)
- Duplicated IDs are solved using Population Figures (at the lower level)
  - NOTE: `admin1` analysis with nationally reported figures should work (verify error handling)
- Total figures (at the lower level) are distributed (to the higher level) using even-population approach
<!-- - Checkout if subnational figures add up to GRID report -->
<!-- - Low resolution figures require new geocoding at the lower level () -->
<!-- - Checkout if subnational figures add up to GRID report
  - Note if levels below `admin_level` were FILTERED before and distribute figures using even-population approach
- TODO: error handling for parent `adm0` if `admin_level` is `adm1`  
-->

In [ ]:
# run only if lower resolution (LR) flows are present
if dest_loc_above + org_loc_above:
    # init acumulator dataframe for figures distributed at LR
    accumulator_LR_df = pd.DataFrame()
    # loop across parent admins
    for i, p_a in enumerate(parent_admins):
        # read population data using selected country/lower admin/year accordingly
        pop_LR_df = pd.read_excel(
            maps_pop_folder + f"{c_iso3}_adm{p_a}_{rep_year}_WorldPop.xlsx"
        )

        # merge population to idmc data at the parent geocoded admin level
        country_idp_LR_df = pd.concat(
            [
                pd.concat(
                    [
                        country_idp_dest_LR_df.query(
                            f"`Locations accuracy`.str.contains('ADM{p_a}')"
                        ) if dest_loc_above else pd.DataFrame(),
                        country_idp_org_LR_df.query(
                            f"`Locations accuracy`.str.contains('ADM{p_a}')"
                        ) if org_loc_above else pd.DataFrame(),
                    ],
                    ignore_index=True,
                ).merge(
                    template_dfs[i],
                    how="left",
                    on=f"adm{p_a + 1}_pcode",
                    validate="m:1",
                ),
                accumulator_LR_df,
            ],
            ignore_index=True,
        ).merge(
            pop_LR_df,
            how="left",
            left_on="parent_pcode",
            right_on=parent_pcode_cols[i],
            validate="m:1",
        )

        # merge country_idp_LR_df with population sums at IDs (for figure distribution at duplicated IDs)
        country_idp_LR_df = country_idp_LR_df.merge(
            country_idp_LR_df.groupby("ID", as_index=False).agg(
                pop_sum_at_ID_LR=(f"tot_pop_{rep_year}", "sum"),
            ),
            how="inner",
            on="ID",
            validate="many_to_one",
        )

        # compute pop weight in new column
        country_idp_LR_df["pop_LR_w"] = (
            country_idp_LR_df[f"tot_pop_{rep_year}"] / country_idp_LR_df.pop_sum_at_ID_LR
        )

        # use pop weight to distribute reported figures with duplicated IDs
        country_idp_LR_df["ID_LR_w"] = (
            country_idp_LR_df.pop_LR_w * country_idp_LR_df["Total figures"]
        )

        # aggregate figures contributed at the LR and merge with LR and admin_level population
        country_agg_LR_df = country_idp_LR_df.groupby(
            "parent_pcode", as_index=False
        ).agg(
            {
                "ID_LR_w": "sum",
                "Figure cause": lambda x: '|'.join(sorted(x.unique())),
                "expand_source": lambda x: '|'.join(sorted(x.unique())),
                # keep ID to use later for the accumulator
                "ID": "first",
            }
        ).merge(
            template_dfs[i],
            how="inner",
            on="parent_pcode",
            validate="1:m",
        ).merge(
            pop_LR_df[[parent_pcode_cols[i], f"tot_pop_{rep_year}"]],
            how="inner",
            left_on="parent_pcode",
            right_on=parent_pcode_cols[i],
            validate="m:1",
        ).merge(
            pd.read_excel(
                maps_pop_folder + f"{c_iso3}_adm{p_a + 1}_{rep_year}_WorldPop.xlsx"
            ).astype(
                {f"adm{p_a + 1}_pcode": str}
            )[[f"adm{p_a + 1}_pcode", f"tot_pop_{rep_year}"]],
            how="inner",
            on=f"adm{p_a + 1}_pcode",
            validate="1:1",
        )

        # distribute figures at LR to next admin_level using even-population approach
        country_agg_LR_df["ID_pop_w"] = (
            country_agg_LR_df.ID_LR_w / country_agg_LR_df[f"tot_pop_{rep_year}_x"] * country_agg_LR_df[f"tot_pop_{rep_year}_y"]
        )

        # acumulate distributed LR figures
        accumulator_LR_df = country_agg_LR_df[
            [
                "ID",
                f"adm{p_a + 1}_pcode",
                "ID_pop_w",
                "Figure cause",
                "expand_source",
            ]
        ].rename(
            columns={
                f"adm{p_a + 1}_pcode": "parent_pcode",
                "ID_pop_w": "Total figures",
            }
        )

        # redefine ID as already de-duplicated (to use in next iteration)
        accumulator_LR_df["ID"] = (
            accumulator_LR_df.ID.astype(str) + accumulator_LR_df.parent_pcode
        )


##### Analysis of National Figures
- Merge Destination with Origin dataframes for national resolution (only one should exist)
  - Assumes national figures can't have duplicated IDs and would be either a Destination or Origin
- Total national figures are distributed to `admin_level` of analysis using even-population approach
<!-- - Checkout if subnational figures add up to GRID report -->
<!-- - Low resolution figures require new geocoding at the lower level () -->
<!-- - Checkout if subnational figures add up to GRID report
  - Note if levels below `admin_level` were FILTERED before and distribute figures using even-population approach -->

In [ ]:
# run only if lower resolution admin0 flows are present
if (
    (dest_loc_above != dest_loc_above_no_nat)
    or (org_loc_above != org_loc_above_no_nat)
):
    # print message
    print(f"NOTE national figures are distributed by population to all admin areas in {c_iso3}_adm{admin_level}_Pcodes_Template")
    # identify nationally reported figures and drop pcode column
    country_idp_nat_df = country_idp_df.query(
        "`Locations accuracy`.str.contains('AM0')"
    ).reset_index(drop=True).drop(columns=[pcode_col])
    # verify IDs are not duplicated
    if country_idp_nat_df.ID.duplicated().sum() != 0:
        raise Exception("Stop execution: national figures shouldn't be duplicated")
    
    # compute pop weight in pop_df new column
    pop_df["pop_w"] = pop_df[f"tot_pop_{rep_year}"] / pop_df[f"tot_pop_{rep_year}"].sum()
    
    # NOTE: there maybe different national figures by `Figure cause`
    # Then, do a cross merge national figures with pop_df
    country_idp_nat_pop_df = country_idp_nat_df.merge(
        pop_df,
        how="cross",
    )
    # distribute national figures to admin_level using even-population approach
    country_idp_nat_pop_df["ID_pop_w"] = (
        country_idp_nat_pop_df.pop_w * country_idp_nat_pop_df["Total figures"]
    )


##### Binarization and checkout
- Aggregate all figures at the selected `admin_level`
- Add Year and Last Updated information
- Binarize output using IDPs rate (over population at Destination)
- Checkout if subnational figures add up to GRID report (or GIDD national updated figures)
<!-- - TODO: add the corresponding info here -->

In [ ]:
# aggregate figures to the selected representation level
country_agg_idp_df = pd.concat(
    [
        country_idp_pop_df,
        country_agg_LR_df[
            [
                pcode_col,
                name_col,
                "ID_pop_w",
                "Figure cause",
                "expand_source",
            ]
        ] if (
            (dest_loc_above + org_loc_above) and (parent_admins)
        ) else pd.DataFrame(),
        country_idp_nat_pop_df[
            [
                pcode_col,
                name_col,
                "ID_pop_w",
                "Figure cause",
                "expand_source",
            ]
        ] if (
            (dest_loc_above != dest_loc_above_no_nat)
            or (org_loc_above != org_loc_above_no_nat)
        ) else pd.DataFrame(),
    ]
).groupby(
    [pcode_col, name_col], as_index=False
).agg(
    {
        "ID_pop_w": "sum",
        "Figure cause": lambda x: ', '.join(sorted(set('|'.join(x).split('|')))),
        "expand_source": lambda x: ', '.join(sorted(set('|'.join(x).split('|')))),
    }
).rename(
    columns={
        "ID_pop_w": "Total_IDPs",
        "expand_source": "Sources",
    }
).merge(
    pop_df,
    how="right",
    on=[pcode_col, name_col],
    validate="1:1",
).fillna(
    {
        "Total_IDPs": 0, "Figure cause": "-", "Sources": "-"
    }
).round(
    {"Total_IDPs": 0}
).astype(
    {"Total_IDPs": int}
)


In [ ]:
# compute IDPs/population rate in new column
# NOTE: save division by zero with fillna zeros
country_agg_idp_df[f"disp_flow_share_{rep_year}"] = (
    country_agg_idp_df.Total_IDPs / country_agg_idp_df[f"tot_pop_{rep_year}"] * 100
).round(4).fillna(0)

# 0/1 binary mapping for wia: below or above mean/median (IDPs rate)
idp_rate_mean = country_agg_idp_df[f"disp_flow_share_{rep_year}"].mean()
idp_rate_med = country_agg_idp_df[f"disp_flow_share_{rep_year}"].median()

# series mapping into new columns
country_agg_idp_df["disp_flow_share_above_mean"] = (
    country_agg_idp_df[f"disp_flow_share_{rep_year}"] >= idp_rate_mean
).astype(int)
country_agg_idp_df["disp_flow_share_above_med"] = (
    country_agg_idp_df[f"disp_flow_share_{rep_year}"] >= idp_rate_med
).astype(int)

# NOTE: assumes impact share has no nulls (should be 0 where no impact)


In [ ]:
# add Last Updated information from json
country_agg_idp_df["Last Updated"] = json_data["lastUpdated"]

# output columns
out_cols = [
    pcode_col,
    name_col,
    f"disp_flow_share_{rep_year}",
    "disp_flow_share_above_mean",
    "disp_flow_share_above_med",
    "Last Updated",
    "Sources",
    "Figure cause",
]


In [ ]:
# save IDPs data
country_agg_idp_df[
    out_cols
].to_excel(
    maps_hazards_folder + f"{c_iso3}_adm{admin_level}_{rep_year}_IDMC.xlsx",
    index=False,
    sheet_name=f"{c_iso3}_adm{admin_level}_{rep_year}",
)


In [ ]:
print(country_agg_idp_df.Total_IDPs.sum())
print(country_agg_idp_df.query("`Figure cause` == 'Conflict'").Total_IDPs.sum())
print(country_agg_idp_df.query("`Figure cause` == 'Disaster'").Total_IDPs.sum())
print(country_agg_idp_df.query("`Figure cause` == 'Conflict, Disaster'").Total_IDPs.sum())

In [ ]:
# Disable automatic rendering
pio.renderers.default = None

color_col = f"disp_flow_share_{rep_year}"
data_min = country_agg_idp_df[color_col].min()
data_max = country_agg_idp_df[color_col].max()
color_max = min(data_max, 100)

# choropleth map of country IDPs share by admin area polygon
disp_flow_share_fig = px.choropleth(
    country_agg_idp_df,
    geojson=json.loads(gdfs[-1].to_json()),
    featureidkey=f"properties.{pcode_col}",
    locations=pcode_col,
    color=color_col,
    range_color=(data_min, color_max),
    color_continuous_scale="YlOrRd",
    projection="mercator",
    width=1200,
    height=900,
)
disp_flow_share_fig.update_geos(fitbounds="locations", visible=False)
disp_flow_share_fig.update_layout(
    title=dict(
        text=f"Internal Displacements during {rep_year} [%] ({c_iso3}_adm{admin_level})",
        x=0.5,  # x position (0-left, 1-right)
        y=0.97,  # y position (0-bottom, 1-top in normalized coordinates)
    ),
    margin={"r": 0, "t": 0, "l": 0, "b": 0}
)
disp_flow_share_fig.update_coloraxes(colorbar_len=0.65)
disp_flow_share_fig.update_traces(marker_line_color="Gainsboro", marker_line_width=0.5)


In [ ]:
# write figure as png direct locally
disp_flow_share_fig.write_image(
    maps_hazards_folder + f"{c_iso3}_adm{admin_level}_{rep_year}_IDMC.png",
    format="png",
    scale=2,
)
